# 📊 Análise Exploratória e Entendimento dos Dados (EDA)
### Desafio Técnico AI / MLOps Engineer — MadeinWeb
**Objetivo:** Compreender a dinâmica do mercado imobiliário de King County (Seattle e região metropolitana), analisar distribuições, tratar anomalias e fundamentar o enriquecimento demográfico por CEP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("Bibliotecas carregadas com sucesso!")

## 1. Carga dos Dados e Verificação Estrutural
Carregamos as três fontes de dados disponibilizadas:
- `kc_house_data.csv`: características físicas dos imóveis residenciais vendidos entre maio/2014 e maio/2015.
- `zipcode_demographics.csv`: indicadores socioeconômicos e demográficos agregados por CEP.
- `future_unseen_examples.csv`: imóveis para os quais o modelo deve prever o preço.

In [ ]:
df_houses = pd.read_csv('../data/raw/kc_house_data.csv')
df_demo = pd.read_csv('../data/raw/zipcode_demographics.csv')
df_unseen = pd.read_csv('../data/raw/future_unseen_examples.csv')

print(f"KC Houses shape: {df_houses.shape}")
print(f"Demographics shape: {df_demo.shape}")
print(f"Future Unseen shape: {df_unseen.shape}")

## 2. Inspeção de Valores Nulos e Tipos de Dados

In [ ]:
print("Nulos em KC Houses:", df_houses.isnull().sum().sum())
print("Nulos em Demographics:", df_demo.isnull().sum().sum())
print("Nulos em Future Unseen:", df_unseen.isnull().sum().sum())

## 3. Análise da Variável Alvo (`price`)
Constatamos forte assimetria positiva (*right-skewed*), variando de US$ 75.000 a US$ 7.700.000, com mediana em US$ 450.000 e média em US$ 540.088.
A aplicação de transformação logarítmica (`log1p`) normaliza a distribuição, estabiliza a variância dos resíduos e evita a penalização desproporcional de mansões nos modelos lineares e de gradiente.

In [ ]:
print(df_houses['price'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_houses['price'], kde=True, ax=axes[0], color='royalblue', bins=40)
axes[0].set_title('Distribuição Original do Preço (Assimetria Positiva)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Preço ($ USD)')

sns.histplot(np.log1p(df_houses['price']), kde=True, ax=axes[1], color='forestgreen', bins=40)
axes[1].set_title('Distribuição com Transformação Log1p (Quase Gaussiana)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log1p(Preço)')
plt.tight_layout()
plt.show()

## 4. Detecção e Tratamento de Anomalias (Outliers de Domínio)
Identificamos a famosa anomalia do dataset King County: **1 imóvel com 33 quartos**, 1.75 banheiros e 1.620 sqft de área construída.
Trata-se de um erro clássico de digitação (o correto é 3 quartos).
Também identificamos pequenas quantidades de imóveis com 0 quartos ou 0 banheiros.

In [ ]:
# Verificação do imóvel com 33 quartos
anomaly = df_houses[df_houses['bedrooms'] == 33]
print(anomaly[['id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'price']])

# Correção
df_houses.loc[df_houses['bedrooms'] == 33, 'bedrooms'] = 3
print("Anomalia corrigida com sucesso!")

## 5. Análise de Correlação das Features Físicas com o Preço

In [ ]:
physical_cols = ['price', 'sqft_living', 'grade', 'sqft_above', 'sqft_living15', 
                 'bathrooms', 'view', 'sqft_basement', 'bedrooms', 'lat', 'long']
corr_matrix = df_houses[physical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', cbar=True)
plt.title('Matriz de Correlação das Variáveis Físicas e Geográficas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Enriquecimento Demográfico e Cobertura de CEPs
Verificamos a cardinalidade dos códigos postais (`zipcode`):
- Treino: 70 CEPs únicos
- Demografia: 70 CEPs únicos (100% de sobreposição)
- Unseen: 45 CEPs únicos (todos contidos na base demográfica)
Desenvolvemos uma estratégia de enriquecimento via *left join* com *fallback* de medianas de condado caso CEPs não mapeados surjam em produção.

In [ ]:
kc_zips = set(df_houses['zipcode'].astype(int))
demo_zips = set(df_demo['zipcode'].astype(int))
unseen_zips = set(df_unseen['zipcode'].astype(int))

print(f"CEPs únicos no treino: {len(kc_zips)}")
print(f"CEPs únicos no demográfico: {len(demo_zips)}")
print(f"CEPs do treino ausentes no demográfico: {len(kc_zips - demo_zips)}")
print(f"CEPs do teste ausentes no demográfico: {len(unseen_zips - demo_zips)}")

## 7. Correlação das Variáveis Demográficas com o Preço dos Imóveis

In [ ]:
df_merged = df_houses.merge(df_demo, on='zipcode', how='left')
demo_corrs = df_merged[['price', 'hous_val_amt', 'medn_incm_per_prsn_amt', 
                        'medn_hshld_incm_amt', 'per_prfsnl', 'per_bchlr', 
                        'per_hsd', 'per_9_to_12']].corr()['price'].sort_values(ascending=False)

print("Correlações dos Indicadores Demográficos com o Preço:")
print(demo_corrs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_merged.sample(2000, random_state=42), x='hous_val_amt', y='price', alpha=0.5, ax=axes[0], color='navy')
axes[0].set_title('Preço do Imóvel vs. Valor Mediano do CEP (hous_val_amt)', fontweight='bold')
axes[0].set_xlabel('Valor Mediano Imóveis do CEP ($)')
axes[0].set_ylabel('Preço de Venda ($)')

sns.scatterplot(data=df_merged.sample(2000, random_state=42), x='medn_hshld_incm_amt', y='price', alpha=0.5, ax=axes[1], color='darkgreen')
axes[1].set_title('Preço do Imóvel vs. Renda Mediana Domiciliar (medn_hshld_incm_amt)', fontweight='bold')
axes[1].set_xlabel('Renda Mediana Domiciliar ($)')
axes[1].set_ylabel('Preço de Venda ($)')
plt.tight_layout()
plt.show()

## 8. Conclusões da EDA
1. **Área útil (`sqft_living`) e acabamento (`grade`)** são os maiores preditores físicos diretos.
2. **Localização e Demografia:** O valor mediano do CEP (`hous_val_amt`), a renda per capita e o percentual com pós-graduação (`per_prfsnl`) exercem enorme influência no preço final do metro quadrado.
3. **Distribuição Espacial:** Imóveis ao norte (próximos a Seattle e Bellevue) possuem valorização significativamente superior a imóveis ao sul do condado.
4. **Decisão de Modelagem:** Modelo deve ser treinado em escala logarítmica com validação cruzada robusta e enriquecimento demográfico acoplado em pipeline.